# MLP for Tabular Medical Data

Goal: Train a small MLP on tabular medical data and compare to a linear baseline.

In [28]:
import sys
sys.path.append("../")  # adds parent folder to Python path
from workshop_utils import load_dataset, basic_train_test_split, build_preprocessor
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

df = load_dataset("cvd")
X_train, X_test, y_train, y_test = basic_train_test_split(df, target="CVD")

num_features = ["age", "chol", "bp"]
cat_features = []
preprocessor = build_preprocessor(num_features, cat_features)

log_clf = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)
log_clf.fit(X_train, y_train)
y_pred_log = log_clf.predict(X_test)
print("Logistic regression performance:")
print(classification_report(y_test, y_pred_log))

mlp_clf = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", MLPClassifier(hidden_layer_sizes=(100, 100), activation="identity", max_iter=1000)),
    ]
)
mlp_clf.fit(X_train, y_train)
y_pred_mlp = mlp_clf.predict(X_test)
print("MLP performance:")
print(classification_report(y_test, y_pred_mlp))

Logistic regression performance:
              precision    recall  f1-score   support

           0       0.54      0.53      0.53        95
           1       0.58      0.60      0.59       105

    accuracy                           0.56       200
   macro avg       0.56      0.56      0.56       200
weighted avg       0.56      0.56      0.56       200

MLP performance:
              precision    recall  f1-score   support

           0       0.54      0.52      0.53        95
           1       0.58      0.60      0.59       105

    accuracy                           0.56       200
   macro avg       0.56      0.56      0.56       200
weighted avg       0.56      0.56      0.56       200



### Experiment: Can you make the MLP outperform logistic regression?

Right now, the **MLP** uses:
- 2 hidden layers, each with 100 nodes  
- the **identity** activation (is that ok?) 
- default regularization and learning rate.

Try adjusting its hyperparameters to see how performance changes.  
Some ideas:

1. **Add nonlinearity**
   - Try different activation functions
   - Does the MLP capture more complex relationships?

2. **Change network size**
   - Try fewer/smaller/more layers: `(8,)`, `(32,16)`, `(64,32,16)`.
   - Too large: overfitting. too small: underfitting.

3. **Adjust regularization**
   - look into documentation to see how its doing that! (alpha) is it L1 or L2? 

4. **Training time**
   - Raise `max_iter` (e.g. `2000`) if you see `ConvergenceWarning`.

5. **Reproducibility**
   - Add `random_state=42` so results are consistent when comparing setups.

6. **Compare**
   - Track accuracy, precision, recall, and F1 in the report.
   - When (if ever) does the MLP actually beat logistic regression?

**Hint:** Logistic regression *is* the optimal model if the data are truly linear.
The MLP can only outperform it if the relationships are nonlinear **and** there’s enough data to learn them.

Take 5-10 minutes to tinker and record what combinations improve test performance!

### Understanding the Metrics

Each metric compares the model’s **predicted labels** to the **true labels**:

- **Accuracy** - “How often did we get it right overall?”  
  - Fraction of all predictions that were correct.

- **Precision** - “When we predicted *disease*, how often were we right?”  
  - High precision = few false alarms (low false positives).

- **Recall (Sensitivity)** - “Of all the people who *actually had disease*, how many did we catch?”  
  - High recall = few missed cases (low false negatives).

- **F1-score** - “Balance between precision and recall.”  
  - Useful when classes are imbalanced or when both false positives and false negatives matter.

**In medical contexts:**  
Missing a true case (low recall) can be worse than a false alarm,  
but a good model aims to balance both.

